# Interactive Refinement - Iterative Human-Agent Collaboration

Many tasks cannot be completed correctly in a single interaction. A first draft of a document rarely matches what someone actually wanted - not because the agent failed, but because requirements only become fully clear through seeing an attempt. Without a structured refinement process these iterative workflows lose context between turns, make it hard to track what has been tried, and risk undoing previously accepted improvements when new feedback is applied.

This notebook demonstrates how to implement interactive refinement using a LangGraph loop. We build a document drafting agent that generates an initial draft, collects structured feedback from a human reviewer, and refines the draft iteratively until the human approves. The key design decision is that every accepted feedback round is stored in history and shown to the agent at the next refinement step, explicitly instructing it to preserve previously approved changes while applying only the new instruction.

In [1]:
import os
from typing import TypedDict, Literal, Optional, List, Dict, Any
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

### Initialize the language model

In [2]:
# Moderate temperature allows creative variation while keeping structure intact
llm = ChatOpenAI(model='gpt-4o-mini', api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.7)

## Understanding interactive refinement
Interactive refinement is fundamentally about controlled iteration. The agent produces a version, the human reviews it and identifies specific changes, the agent applies those changes and produces a new version, and this cycle continues until the result matches the human's intent. What distinguishes a well-engineered refinement loop from an ad-hoc conversation is structure: each feedback round is categorised, each accepted change is recorded, and every new version is generated with full awareness of the entire change history.

The most important architectural property of a refinement loop is regression prevention. If the human approved a specific phrasing in iteration 2, the agent should not undo that phrasing in iteration 3 while addressing a different piece of feedback. Achieving this requires passing the full feedback history into the refinement prompt so the agent can see what has already been accepted and constrain its changes accordingly.

LangGraph implements the loop through a conditional edge that routes the feedback node either back to the generation node for another pass, or to `END` when the human approves. The loop condition lives in a single routing function, making it easy to add termination conditions such as a maximum iteration count.

### Define the refinement state
The state accumulates information across iterations. Input fields are set once at invocation. Tracking fields grow with each pass through the loop. Feedback fields are overwritten each iteration with the latest input, while the history fields preserve every round.

In [3]:
class RefinementState(TypedDict):
    '''State for the interactive refinement workflow.

    Accumulates history across iterations: version_history records every draft generated, feedback_history records every accepted change. Both grow monotonically and are never overwritten.
    '''
    # Input -- set once at invocation, never modified
    task_description: str
    requirements: Optional[Dict[str, Any]]   # Constraints the draft must satisfy

    # Iteration tracking -- updated by generate_draft each pass
    iteration: int
    current_version: Optional[str]           # The most recent draft
    version_history: List[Dict]              # {iteration, content} for every draft
    feedback_history: List[Dict]             # {iteration, type, text} for every accepted change

    # Current feedback -- written by collect_feedback, read by generate_draft
    feedback_type: Optional[Literal['content', 'structure', 'tone', 'expand', 'reduce', 'approve']]
    feedback_text: Optional[str]             # The specific instruction to apply

    # Completion -- flipped to True by collect_feedback on approval
    is_complete: bool

The `RefinementState` groups fields into four logical zones, each owned by a specific part of the workflow.
- The input fields (`task_description`, `requirements`) are set at invocation and never modified by any node. They provide stable context for every generation call across all iterations.
- `version_history` and `feedback_history` grow monotonically: every node appends to them using the `state.get(..., []) + [new_record]` pattern rather than mutating in place. This is the correct approach for LangGraph state updates.
- `feedback_type` and `feedback_text` are overwritten each iteration with the latest values from `collect_feedback`. They carry the current instruction into `generate_draft`, where the instruction is acted on and then appended to `feedback_history`.
- `is_complete` is the loop termination signal. It starts as `False` and is flipped to `True` only by the approval path in `collect_feedback`. The routing function reads this field to decide whether to loop or exit.

### Generate an initial draft or a refined version
The generation node has two distinct paths: an initial-draft path used only on the first iteration, and a refinement path for all subsequent ones. The critical feature of the refinement path is that it shows `feedback_history` - all previously accepted changes - to the LLM alongside the new instruction, explicitly telling the model what to preserve while applying the change.

In [4]:
def generate_draft(state: RefinementState) -> dict:
    '''Generate the initial draft or a refined version based on feedback history.

    On the refinement path, both past accepted changes and the new instruction are shown to the LLM. Separating them in the prompt lets the model distinguish what to preserve from what to change.
    '''
    task      = state['task_description']
    reqs      = state.get('requirements') or {}
    iteration = state.get('iteration', 0)

    if iteration == 0:
        # Format requirements as a readable list; default to 'none specified' if empty
        req_text = '\n'.join(f'- {k}: {v}' for k, v in reqs.items()) if reqs else 'none specified'
        prompt_lines = [
            f'Write the following: {task}',
            '',
            'Requirements:',
            req_text,
            '',
            'Produce a complete, polished draft.',
        ]

    else:
        # Refinement: show the current draft, past accepted changes, and the new instruction
        current      = state['current_version']
        past_feedback = state.get('feedback_history', [])
        new_type     = state.get('feedback_type', 'content')
        new_text     = state.get('feedback_text', '')

        prompt_lines = [
            'Refine the following draft. Apply only the new instruction below.',
            'Preserve sections that previous feedback already approved.',
            '',
            'CURRENT DRAFT:',
            current,
            '',
        ]

        # Show previously accepted changes so the model knows what must be preserved
        if past_feedback:
            prompt_lines.append('PREVIOUSLY ACCEPTED CHANGES (do not reverse these):')
            for fb in past_feedback:
                fb_type = fb['type']
                fb_text = fb['text']
                prompt_lines.append(f'  [{fb_type}] {fb_text}')
            prompt_lines.append('')

        prompt_lines += [
            f'NEW INSTRUCTION ({new_type}) -- apply this change now:',
            new_text,
        ]

    prompt = '\n'.join(prompt_lines)
    result = llm.invoke(prompt)
    new_version = result.content

    version_record = {
        'iteration': iteration + 1,
        'content': new_version,
    }

    # Record the just-applied feedback in history so future iterations can preserve it.
    # Skip on iteration 0 (initial draft has no feedback to record).
    past = state.get('feedback_history', [])
    if state.get('feedback_text'):
        feedback_text = state['feedback_text']
        feedback_type = state.get('feedback_type', 'content')
        accepted = {'iteration': iteration, 'type': feedback_type, 'text': feedback_text}
        updated_history = past + [accepted]
    else:
        updated_history = past

    label   = '\U0001f4dd Initial draft' if iteration == 0 else f'\u2728 Revision {iteration + 1}'
    preview = new_version[:500] + '...' if len(new_version) > 500 else new_version
    print(f'\n{label}')
    print('-' * 65)
    print(preview)
    print('=' * 65)

    return {
        'iteration': iteration + 1,
        'current_version': new_version,
        'version_history': state.get('version_history', []) + [version_record],
        'feedback_history': updated_history,
    }

`generate_draft` implements the two paths with different prompt structures.
- The initial-draft prompt formats `requirements` as a readable bullet list using a generator expression. An empty or absent requirements dict produces `'none specified'` rather than an unhelpful empty string or raw `{}`.
- The refinement prompt explicitly separates two concerns: a `PREVIOUSLY ACCEPTED CHANGES` block that tells the model what to preserve, and a `NEW INSTRUCTION` block that tells it what to change. This separation is what prevents regressions - the model sees a clear boundary between what has been approved and what is being asked for now.
- The `feedback_history` update happens in `generate_draft`, not in `collect_feedback`. This means the history contains feedback that has been acted on, not feedback that is merely pending. On iteration 0 the guard `if state.get('feedback_text')` is `False`, so no spurious record is appended.
- Both the `version_history` and `feedback_history` updates use the `existing + [new_record]` pattern, producing a new list rather than mutating the existing one.

### Collect refinement feedback
After the draft is displayed, the feedback node pauses and waits for the human to classify the type of change needed and describe what to change. An `approve` selection signals the end of the session by setting `is_complete` to `True`, which the routing function reads when deciding whether to loop or exit.

In [5]:
def collect_feedback(state: RefinementState) -> dict:
    '''Collect the human's feedback type and text, or their approval.

    Does not update feedback_history -- that is done by generate_draft after the feedback has been acted on. This node only captures the current instruction and the loop termination signal.
    '''
    iteration = state.get('iteration', 0)

    print('\n' + '=' * 65)
    print(f'  FEEDBACK \u2014 iteration {iteration}')
    print('=' * 65)
    print('\n  Choose feedback type:')
    print('    1  Content   \u2014 change what is said')
    print('    2  Structure \u2014 reorganise sections or flow')
    print('    3  Tone      \u2014 adjust style or formality')
    print('    4  Expand    \u2014 add more detail or examples')
    print('    5  Reduce    \u2014 make it more concise')
    print('    6  Approve   \u2014 done, no more changes needed')

    choice = input('\n  Enter 1-6: ').strip()

    type_map = {
        '1': 'content',
        '2': 'structure',
        '3': 'tone',
        '4': 'expand',
        '5': 'reduce',
        '6': 'approve',
    }
    feedback_type = type_map.get(choice, 'content')  # default to 'content' for invalid input

    if feedback_type == 'approve':
        print('\n  \u2713 Approved \u2014 refinement complete')
        return {
            'feedback_type': 'approve',
            'feedback_text': None,
            'is_complete': True,
        }

    # Collect the specific instruction; this will be passed to generate_draft
    text = input(f'\n  Describe the {feedback_type} change needed:\n  > ').strip()

    print(f'\n  Feedback recorded: {feedback_type}')
    return {
        'feedback_type': feedback_type,
        'feedback_text': text,
        'is_complete': False,
    }

`collect_feedback` keeps the human interaction minimal and structured.
- Six feedback categories cover the most common types of document revision. Using a numbered menu ensures `feedback_type` always receives a known string rather than a potentially ambiguous free-text value that would require classification.
- The `type_map.get(choice, 'content')` default means invalid input (anything other than 1-6) falls back to `'content'` rather than crashing or prompting again. This is appropriate for a demo notebook where usability is more important than strict validation.
- `feedback_history` is deliberately not updated here. The update happens in `generate_draft` after the feedback has been acted on. This keeps the history semantically accurate: it records changes that were implemented, not changes that are merely pending.
- The approval path returns `feedback_text: None`. The generation node guards against reading this with `if state.get('feedback_text')`, so a `None` value is handled cleanly without conditional checks at the call site.

### Build the refinement loop graph
The graph implements the iterative refinement pattern with a single conditional edge that loops back to the generation node. This is the minimal structure needed: generation is always followed by feedback collection, which then either exits or repeats the cycle. The loop can run for as many iterations as the human requires.

In [6]:
def route_after_feedback(state: RefinementState) -> str:
    '''Exit when approved; loop back for another refinement pass otherwise.'''
    if state.get('is_complete', False):
        return 'done'
    return 'generate'


# Build the graph
workflow = StateGraph(RefinementState)

workflow.add_node('generate', generate_draft)
workflow.add_node('feedback', collect_feedback)

# Every iteration: generate a draft, then collect feedback on it
workflow.set_entry_point('generate')
workflow.add_edge('generate', 'feedback')

# The conditional edge is what makes this a loop:
# 'generate' routes back to the generation node for another pass;
# 'done' maps to END and terminates the workflow.
workflow.add_conditional_edges(
    'feedback',
    route_after_feedback,
    {
        'generate': 'generate',  # loop back for another refinement pass
        'done':     END,         # human approved -- workflow ends
    }
)

refinement_loop = workflow.compile()

The graph structure makes the refinement loop explicit and inspectable in the compiled graph object.
- The unconditional edge from `generate` to `feedback` ensures every generated draft is reviewed before any routing decision is made.
- The `'generate' -> 'generate'` loop-back is what distinguishes a refinement graph from a simple chain. LangGraph supports this through conditional edges that return the name of any registered node, including the current one or a previous one. The loop runs until `route_after_feedback` returns `'done'`.
- `route_after_feedback` reads a single boolean field, making it the simplest possible routing function. All loop termination logic lives in `collect_feedback` (which sets `is_complete`); the router just reads the result.
- For production use, add a maximum iteration guard to `route_after_feedback` - for example, `if state.get('iteration', 0) >= 10: return 'done'` - so the loop cannot run indefinitely if the human never approves.

### Run an interactive refinement session
The initial state seeds all history fields as empty lists, all feedback fields as `None`, and `iteration` at zero. As the session progresses, `iteration` increments, `version_history` accumulates every draft, and `feedback_history` accumulates every accepted change. The session ends when the human enters `6` (approve) at the feedback prompt.

In [7]:
initial_state: RefinementState = {
    'task_description': 'Write a product announcement email for an AI writing assistant',
    'requirements': {'length': '200-300 words', 'tone': 'professional'},
    # Iteration tracking -- starts empty, populated by generate_draft
    'iteration': 0,
    'current_version': None,
    'version_history': [],
    'feedback_history': [],
    # Feedback fields -- written each iteration by collect_feedback
    'feedback_type': None,
    'feedback_text': None,
    # Completion -- flipped to True when the human approves
    'is_complete': False,
}

print('\U0001f680 Starting interactive refinement session')
print('   Enter feedback after each draft; choose 6 to approve and finish.\n')
result = refinement_loop.invoke(initial_state)

🚀 Starting interactive refinement session
   Enter feedback after each draft; choose 6 to approve and finish.


📝 Initial draft
-----------------------------------------------------------------
Subject: Introducing Your New AI Writing Assistant – Elevate Your Writing Efforts!

Dear [Recipient's Name],

We are thrilled to announce the launch of our cutting-edge AI Writing Assistant, designed to revolutionize the way you create, edit, and enhance your written content. As part of our commitment to empowering professionals across various industries, this innovative tool is engineered to streamline your writing process while maintaining the highest standards of quality.

Our AI Writing Assi...

  FEEDBACK — iteration 1

  Choose feedback type:
    1  Content   — change what is said
    2  Structure — reorganise sections or flow
    3  Tone      — adjust style or formality
    4  Expand    — add more detail or examples
    5  Reduce    — make it more concise
    6  Approve   — done, no more 


  Enter 1-6:  2

  Describe the structure change needed:
  >  More casual



  Feedback recorded: structure

✨ Revision 2
-----------------------------------------------------------------
Subject: Meet Your New AI Writing Buddy – Let’s Boost Your Writing Together!

Hey [Recipient's Name],

We’re super excited to introduce you to our brand-new AI Writing Assistant! This tool is here to shake up how you create, edit, and polish your written content. We’re all about helping professionals like you across different fields, and this cool innovation is designed to make your writing process smoother while keeping your work top-notch.

Our AI Writing Assistant uses cutting-edge natural la...

  FEEDBACK — iteration 2

  Choose feedback type:
    1  Content   — change what is said
    2  Structure — reorganise sections or flow
    3  Tone      — adjust style or formality
    4  Expand    — add more detail or examples
    5  Reduce    — make it more concise
    6  Approve   — done, no more changes needed



  Enter 1-6:  6



  ✓ Approved — refinement complete


## Examining the refinement history
Once a session completes, the full trajectory of the refinement is available in the result state. The `version_history` list shows every draft that was generated with its iteration number and length. The `feedback_history` list shows every round of feedback that was accepted - the type, the specific instruction, and which iteration it came from - in the order it was applied.

This history serves two purposes: in the current session it was injected into the refinement prompt to prevent regressions; in a production system it could be mined for patterns to improve future initial drafts, reducing the average number of iterations needed to reach approval.

In [8]:
# Examine the refinement trajectory after the session completes
print('\n' + '=' * 65)
print('  REFINEMENT SUMMARY')
print('=' * 65)
print(f'  Total iterations : {result["iteration"]}')
print(f'  Feedback rounds  : {len(result["feedback_history"])}')

if result['version_history']:
    print('\n  Version history:')
    for v in result['version_history']:
        v_num  = v['iteration']
        v_chars = len(v['content'])
        print(f'    v{v_num} : {v_chars} characters')

if result['feedback_history']:
    print('\n  Feedback applied:')
    for fb in result['feedback_history']:
        fb_iter    = fb['iteration']
        fb_type    = fb['type']
        fb_text    = fb['text']
        fb_preview = fb_text[:60] + '...' if len(fb_text) > 60 else fb_text
        print(f'    [{fb_iter}] {fb_type}: {fb_preview}')

print('\n  Final version:')
final = result.get('current_version') or ''
print(final[:300] + '...' if len(final) > 300 else final)


  REFINEMENT SUMMARY
  Total iterations : 2
  Feedback rounds  : 1

  Version history:
    v1 : 1801 characters
    v2 : 1726 characters

  Feedback applied:
    [1] structure: More casual

  Final version:
Subject: Meet Your New AI Writing Buddy – Let’s Boost Your Writing Together!

Hey [Recipient's Name],

We’re super excited to introduce you to our brand-new AI Writing Assistant! This tool is here to shake up how you create, edit, and polish your written content. We’re all about helping professional...


The refinement session produced a document through structured iteration where each round of feedback built on the previous ones. The `feedback_history` field was the mechanism that made this possible: by injecting it into the refinement prompt as a list of changes to preserve, the agent could apply new instructions without undoing previously accepted content.

When to use interactive refinement:
- Document or content creation where requirements only become clear through seeing an attempt.
- Design tasks where the human refines their vision through iterative feedback on concrete drafts.
- Complex query formulation for search, reporting, or data extraction where the first attempt reveals what is and is not needed.
- Any task where first-attempt quality is insufficient but the gap can be closed through a bounded number of structured feedback rounds.

Design principles that matter in production:
- Include the full feedback history in every refinement prompt - showing only the latest feedback causes the agent to treat each iteration as a fresh start and undo previously accepted improvements.
- Record feedback in `generate_draft` after acting on it, not in `collect_feedback` before - this keeps `feedback_history` semantically accurate as a record of implemented changes rather than pending instructions.
- Use the conditional loop-back edge for termination rather than building termination logic into the generation node - this keeps the routing decision visible in the graph structure and easy to add conditions to.
- Add a maximum iteration guard in `route_after_feedback` for production deployments so the loop cannot run indefinitely if the human never approves.

In production, the accumulated `feedback_history` records across many sessions can train a preference model: if certain feedback types appear consistently in the first iteration across many users, the initial-draft prompt should be adjusted to address those patterns before they are requested, gradually reducing the average number of iterations needed to reach approval.